# Contexto Kedro

In [15]:
import os
import pandas as pd
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[05/15/26 10:40:51] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=10223;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=727400;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\UNIS\unis-perm-flow


In [16]:
# ---------- 1. CONFIGURACIÓN Y RUTAS ----------
input_path = "G:/Unidades compartidas/Alianzas/3. Data/UNIS/"
output_path = "G:/Unidades compartidas/Alianzas/3. Data/UNIS/"
fecha_hoy = datetime.now().strftime("%Y-%m-%d")
fecha_actual = datetime.now()
hoy = pd.to_datetime(fecha_actual)
fecha_str = fecha_actual.strftime("%Y-%m-%d")

In [17]:
# Función para limpiar códigos (quita espacios y '.0' de decimales)
def limpiar_formato(valor):
    s = str(valor).strip()
    if s.endswith('.0'):
        return s[:-2]
    return s

In [18]:
# ---------- 2. LECTURA DE DATOS (HISTÓRICO) ----------
print("📂 Cargando archivos base...")

# Cargar Ciclos
archivo_ciclos = os.path.join(input_path, "Base Rematricula/input/Ciclo Rematricula UNIS.xlsx")
df_ciclos = pd.read_excel(archivo_ciclos)

📂 Cargando archivos base...


In [19]:
archivo_ciclos

'G:/Unidades compartidas/Alianzas/3. Data/UNIS/Base Rematricula/input/Ciclo Rematricula UNIS.xlsx'

In [13]:
# Cargar Inscripciones (header=1)
archivo_inscripciones = os.path.join(input_path, "Base Rematricula/input/UNIS_INSCRIPCIONES_FACULTAD_PU_29830.xlsx")
df_raw = pd.read_excel(archivo_inscripciones, header=1)
df_raw.columns = df_raw.columns.str.strip() # Limpiar nombres de columnas

[05/15/26 10:36:08] WARNING  c:\Users\User\miniconda3\envs\unis\Lib\site-packages\openpyxl\styles\s ]8;id=968513;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py\warnings.py]8;;\:]8;id=184783;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py#110\110]8;;\
                             tylesheet.py:237: UserWarning: Workbook contains no default style,                    
                             apply openpyxl's default                                                              
                               warn("Workbook contains no default style, apply openpyxl's default")                
                                                                                                                   

In [14]:
# --- DIAGNÓSTICO DE COLUMNAS (IMPORTANTE) ---
# Verificamos que la columna clave exista
if 'Ccl Admis' not in df_raw.columns:
    print("❌ ERROR: No se encuentra la columna 'Ccl Admis' en el archivo de inscripciones.")
    print("   Columnas encontradas:", df_raw.columns.tolist())
    exit()

In [ ]:
# --- LIMPIEZA DE DATOS PARA EL CRUCE ---
print("🧹 Normalizando códigos de cohorte...")

# Aplicar limpieza a ambos lados para asegurar coincidencia
df_ciclos['Cuatrimestre'] = df_ciclos['Cuatrimestre'].apply(limpiar_formato)
df_raw['Ccl Admis'] = df_raw['Ccl Admis'].apply(limpiar_formato)

# Muestra de qué estamos comparando (Para depuración)
print(f"   Ejemplo Ciclos: {df_ciclos['Cuatrimestre'].unique()[:5]}")
print(f"   Ejemplo Inscripciones: {df_raw['Ccl Admis'].unique()[:5]}")

# ---------- 3. PROCESAMIENTO DE HISTORIAL ----------
print("⚙️ Procesando historial de inscripciones...")
historial = []

for _, fila in df_ciclos.iterrows():
    cohorte = fila["Cuatrimestre"]
    
    # Filtrar estudiantes
    df_filtrado = df_raw[df_raw["Ccl Admis"] == cohorte].copy()
    
    if df_filtrado.empty:
        continue
    
    # Renombrar columnas
    df_hist = df_filtrado.rename(columns={
        "Número de Documento": "documento",
        "Nombre estudiante": "nombre",
        "Correo-E": "correo",
        "Prog Acad": "programa",
        "Ccl Admis": "periodo_ingreso",
        "Ciclo Lectivo": "ciclo",
        "Estado.1": "estado_ciclo"
    })
    
    cols_a_mantener = [
        "documento", "nombre", "correo", "programa",
        "periodo_ingreso", "ciclo", "estado_ciclo"
    ]
    
    # Asegurar que las columnas existen antes de filtrar
    cols_existentes = [c for c in cols_a_mantener if c in df_hist.columns]
    df_hist = df_hist[cols_existentes].drop_duplicates()
    
    historial.append(df_hist)

# Validar si se generaron datos
if historial:
    df_historial_final = pd.concat(historial, ignore_index=True)
    print(f"✅ Historial generado con {len(df_historial_final)} registros.")
else:
    print("⚠️ CRÍTICO: No hubo coincidencias entre 'Ccl Admis' y 'Cuatrimestre'.")
    print("   Revisa los 'Ejemplos' impresos arriba para ver por qué no coinciden.")
    exit()

# Limpieza final del histórico
df_historial_final = df_historial_final.drop_duplicates(
    subset=["documento", "periodo_ingreso", "ciclo", "estado_ciclo"]
)

# ---------- 4. INTEGRACIÓN STUDENT JOURNEY ----------
print("🚀 Integrando Student Journey...")

archivo_fechas = os.path.join(input_path, 'Base Estados/input/Student Journey UNIS.xlsx')
df_fechas = pd.read_excel(archivo_fechas)
df_fechas.columns = df_fechas.columns.str.strip()

# Normalizar columna clave en Student Journey también
if 'Cohorte' in df_fechas.columns:
    df_fechas['Cohorte'] = df_fechas['Cohorte'].apply(limpiar_formato)
else:
    print("❌ No se encontró la columna 'Cohorte' en el Student Journey")
    exit()

# Convertir fechas
fecha_cols = [
    "Pre welcome_inicio", "Pre welcome_fin",
    "Onboarding_inicio", "Onboarding_fin",
    "Q1_inicio", "Q1_fin", "QA_inicio"
]
for col in fecha_cols:
    if col in df_fechas.columns:
        df_fechas[col] = pd.to_datetime(df_fechas[col], errors='coerce')

# MERGE
df_completo = pd.merge(
    df_historial_final, 
    df_fechas, 
    left_on='periodo_ingreso', 
    right_on='Cohorte', 
    how='left'
)

# ---------- 5. CALCULO DE FASE ----------
def determinar_fase(row):
    if pd.isnull(row.get('Pre welcome_inicio')):
        return "Sin información de fechas"
    
    if hoy < row["Pre welcome_inicio"]:
        return "Antes de inicio"
    elif row["Pre welcome_inicio"] <= hoy <= row["Pre welcome_fin"]:
        return "Pre welcome"
    elif row["Onboarding_inicio"] <= hoy <= row["Onboarding_fin"]:
        return "Onboarding"
    elif row["Q1_inicio"] <= hoy <= row["Q1_fin"]:
        return "Q1"
    elif row.get("QA_inicio") and hoy >= row["QA_inicio"]:
        return "QA o posterior"
    else:
        return "Fuera de rango"

df_completo['Fase actual'] = df_completo.apply(determinar_fase, axis=1)
df_completo['fecha_ejecucion'] = fecha_str


In [ ]:
# ---------- 6. EXPORTACIÓN ----------
cols_finales = [
    "documento", "nombre", "correo", "programa", 
    "periodo_ingreso", "ciclo", "estado_ciclo", 
    "Fase actual", "fecha_ejecucion"
]
cols_finales = [c for c in cols_finales if c in df_completo.columns]

ruta_salida_principal = os.path.join(output_path, "Base Historico/output/Inscripciones_Materias.xlsx")
ruta_salida_principal_hist = os.path.join(output_path, f"Base Historico/output/historico/Inscripciones_Materias_{fecha_hoy}.xlsx")
os.makedirs(os.path.dirname(ruta_salida_principal), exist_ok=True)

df_completo[cols_finales].to_excel(ruta_salida_principal, index=False)
df_completo[cols_finales].to_excel(ruta_salida_principal_hist, index=False)
print(f"✅ Archivo final exportado: {ruta_salida_principal}")

# Copia local
output_local = "C:/Users/User/Documents/UNIS"
output_local_hist = "C:/Users/User/Documents/UNIS/historico"
if os.path.exists(output_local):
    ruta_salida_local = os.path.join(output_local, "Inscripciones_Materias.xlsx")
    ruta_salida_local_hist = os.path.join(output_local_hist, f"Inscripciones_Materias_{fecha_hoy}.xlsx")
    df_completo[cols_finales].to_excel(ruta_salida_local, index=False)
    df_completo[cols_finales].to_excel(ruta_salida_local_hist, index=False)
    print(f"📂 Copia local guardada.")